# 18: Export the website models to Hugging Face

Packages the detectors used by the website and uploads them to the model repository `tanish-237/ai-image-forensics-models`:

- `m1.pt`: ResNet-50 from E8 (half precision)
- `m2_lr.npz`: the logistic-regression weights on CLIP features from E8 (re-fitted from the saved features; the fit is deterministic)
- `attributor.pt`: the seven-class attribution model from E4
- `config.json`: which detector gives the verdict, the calibrated thresholds and the names shown on the page

**Calibration.** E8 showed that averaging tile scores over the full image ranks images well but lowers the scores, so the threshold 0.5 no longer fits. The thresholds are therefore chosen on 600 full-size images from OpenFake test file 3, which no experiment has used: 300 real (DOCCI and ImageNet) and 300 generated. The threshold that maximises balanced accuracy is kept, and the WhatsApp images are used as an independent check.

You need a Hugging Face **write** token (huggingface.co, Settings, Access Tokens, New token, type *Write*). It is typed into a hidden prompt and not saved in the notebook. GPU runtime recommended.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import json
from getpass import getpass
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import pyarrow.parquet as pq
import torch.nn as nn
import torch.nn.functional as F
from huggingface_hub import HfApi, hf_hub_download
from PIL import Image
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import log_loss, roc_auc_score, roc_curve
from torchvision.models import resnet50
from transformers import CLIPVisionModelWithProjection
import io

ROOT = Path("/content/drive/MyDrive/minor-project")
DATA = ROOT / "processed_openfake_train_full"
OUT = Path("/content/website_models")
OUT.mkdir(exist_ok=True)

Mounted at /content/drive


## Settings

In [2]:
VARIANT = "full"         # "full" (full data) or "full_aug" (full data + augmentation); E8: "full" was better
MAIN = "M2"              # detector that gives the verdict: "M1", "M2" or "mean"
REPO = "tanish-237/ai-image-forensics-models"
PRIVATE = False
CALIBRATION_FILE = "core/test-00003-of-00013.parquet"
PER_CLASS = 300
SIZE = 224
MAX_TILES = 9
SEED = 42
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

## Package the models

In [3]:
def save_half(src, dst):
    state = torch.load(src, map_location="cpu")
    torch.save({k: v.half() if v.is_floating_point() else v for k, v in state.items()}, dst)


save_half(ROOT / "checkpoints" / ("e8_m1_full_aug.pt" if VARIANT == "full_aug" else "e8_m1_full.pt"), OUT / "m1.pt")
save_half(ROOT / "checkpoints" / "e4_a2_resnet50.pt", OUT / "attributor.pt")

meta = pd.read_csv(DATA / "meta.csv")
y = (meta["label"] == "fake").astype(int).to_numpy()
train, val = (meta["split"] == "train").to_numpy(), (meta["split"] == "val").to_numpy()
X, Y = np.load(DATA / "clip_train.npy"), y[train]
if VARIANT == "full_aug":
    X = np.concatenate([X, np.load(DATA / "clip_train_aug1.npy"), np.load(DATA / "clip_train_aug2.npy")])
    Y = np.concatenate([Y] * 3)
f_val = np.load(DATA / "clip_val.npy")
search = [(log_loss(y[val], LogisticRegression(C=C, max_iter=5000).fit(X, Y).predict_proba(f_val)[:, 1]), C) for C in [0.1, 1, 10, 100]]
C = min(search)[1]
clf = LogisticRegression(C=C, max_iter=5000).fit(X, Y)
np.savez(OUT / "m2_lr.npz", coef=clf.coef_.astype(np.float32), intercept=np.array(clf.intercept_[0], dtype=np.float32))
print("M2 regularisation C =", C)

label = "trained on recent generators" + (", with augmentation" if VARIANT == "full_aug" else "")
config = {
    "main": MAIN,
    "clip_name": "openai/clip-vit-large-patch14",
    "sources": ["Real", "ADM", "GLIDE", "Midjourney", "SD15", "VQDM", "Wukong"],
    "names": {"M1": f"ResNet-50 ({label})", "M2": f"CLIP + linear classifier ({label})"},
    "description": f"E8 detectors ({VARIANT}); verdict from {MAIN}",
}
(OUT / "config.json").write_text(json.dumps(config, indent=2))
(OUT / "README.md").write_text(
    "---\nlicense: cc-by-nc-4.0\n---\n\n# AI Image Forensics models\n\n"
    "Detectors used by the website of our B.Tech minor project (MNNIT Allahabad, 2026). "
    "Trained on part of the OpenFake training split (CC BY-NC 4.0); the attribution model on Tiny GenImage (CC BY-NC-SA 4.0). "
    "For non-commercial research and teaching only.\n"
)
for f in sorted(OUT.iterdir()):
    print(f"{f.name}: {f.stat().st_size / 1e6:.1f} MB")

M2 regularisation C = 10
README.md: 0.0 MB
attributor.pt: 47.2 MB
config.json: 0.0 MB
m1.pt: 47.2 MB
m2_lr.npz: 0.0 MB


## Calibrate the full-image thresholds

The same tile grid as the website: up to 3 x 3 tiles of 224 x 224 pixels at original resolution, scores averaged.

In [4]:
def tile_positions(width, height):
    per_side = int(np.sqrt(MAX_TILES))
    xs = np.unique(np.linspace(0, width - SIZE, min(per_side, max(1, width // SIZE))).astype(int) // 16 * 16)
    ys = np.unique(np.linspace(0, height - SIZE, min(per_side, max(1, height // SIZE))).astype(int) // 16 * 16)
    return [(x, y) for y in ys for x in xs]


def tiles_of(img):
    return np.stack([np.asarray(img.crop((x, y, x + SIZE, y + SIZE))) for x, y in tile_positions(*img.size)])


m1 = resnet50(weights=None)
m1.fc = nn.Linear(m1.fc.in_features, 1)
m1.load_state_dict({k: v.float() for k, v in torch.load(OUT / "m1.pt").items()})
m1 = m1.to(device).eval()
dtype = torch.float16 if device.type == "cuda" else torch.float32
clip = CLIPVisionModelWithProjection.from_pretrained(config["clip_name"], torch_dtype=dtype).to(device).eval()
w = torch.from_numpy(clf.coef_[0]).float().to(device)
b = float(clf.intercept_[0])
MEAN = torch.tensor([0.485, 0.456, 0.406], device=device).view(1, 3, 1, 1)
STD = torch.tensor([0.229, 0.224, 0.225], device=device).view(1, 3, 1, 1)
CLIP_MEAN = torch.tensor([0.48145466, 0.4578275, 0.40821073], device=device).view(1, 3, 1, 1)
CLIP_STD = torch.tensor([0.26862954, 0.26130258, 0.27577711], device=device).view(1, 3, 1, 1)


@torch.no_grad()
def full_image_scores(img):
    x = torch.from_numpy(tiles_of(img.convert("RGB"))).to(device).permute(0, 3, 1, 2).float() / 255
    p1 = torch.sigmoid(m1((x - MEAN) / STD).squeeze(1)).mean().item()
    emb = F.normalize(clip(pixel_values=((x - CLIP_MEAN) / CLIP_STD).to(dtype)).image_embeds.float(), dim=1)
    p2 = torch.sigmoid(emb @ w + b).mean().item()
    return {"M1": p1, "M2": p2, "mean": (p1 + p2) / 2}


path = hf_hub_download("ComplexDataLab/OpenFake", CALIBRATION_FILE, repo_type="dataset", local_dir="/content/openfake_calibration")
pf = pq.ParquetFile(path)
labels_table = pf.read(columns=["label", "model"]).to_pandas()
rng = np.random.default_rng(SEED)
chosen = set()
for label in ["real", "fake"]:
    idx = labels_table.index[labels_table["label"] == label].to_numpy()
    chosen |= set(rng.choice(idx, size=min(len(idx), PER_CLASS * 2), replace=False).tolist())

rows, start, counts = [], 0, {"real": 0, "fake": 0}
for g in range(pf.num_row_groups):
    n = pf.metadata.row_group(g).num_rows
    wanted = [r for r in range(start, start + n) if r in chosen]
    if wanted:
        t = pf.read_row_group(g, columns=["image", "label", "model"]).to_pylist()
        for r in wanted:
            item = t[r - start]
            img = Image.open(io.BytesIO(item["image"]["bytes"]))
            if img.mode in ("L", "LA", "1", "I", "I;16") or min(img.size) < SIZE or counts[item["label"]] >= PER_CLASS:
                continue
            counts[item["label"]] += 1
            rows.append({"source": item["model"], "y": int(item["label"] == "fake"), **full_image_scores(img)})
    start += n
calibration = pd.DataFrame(rows)
calibration.to_csv(ROOT / "results" / "website_calibration.csv", index=False)
print(calibration.groupby("y").size().to_dict())

thresholds = {}
for m in ["M1", "M2", "mean"]:
    fpr, tpr, thr = roc_curve(calibration["y"], calibration[m])
    finite = np.isfinite(thr)
    fpr, tpr, thr = fpr[finite], tpr[finite], thr[finite]
    best = np.argmax(tpr - fpr)
    thresholds[m] = float(np.clip(thr[best], 0.01, 0.99))
    print(f"{m}: AUC {roc_auc_score(calibration['y'], calibration[m]):.3f}, threshold {thresholds[m]:.3f} "
          f"(real called real {1 - fpr[best]:.1%}, AI detected {tpr[best]:.1%})")
config["thresholds"] = thresholds
(OUT / "config.json").write_text(json.dumps(config, indent=2))

config.json:   0%|          | 0.00/4.52k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.71GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/392 [00:00<?, ?it/s]

[transformers] CLIPVisionModelWithProjection LOAD REPORT from: openai/clip-vit-large-patch14
Key                                                          | Status     |  | 
-------------------------------------------------------------+------------+--+-
text_model.encoder.layers.{0...11}.self_attn.out_proj.weight | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.k_proj.bias     | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.mlp.fc1.weight            | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.k_proj.weight   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.q_proj.weight   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.v_proj.weight   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.layer_norm2.weight        | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.mlp.fc2.bias              | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.layer_norm2.bias          | UNEXPECTED |  | 
text_model.encoder.layers.{

core/test-00003-of-00013.parquet: reconstructing file:   0%|          |  0.00B / 5.24GB            

core/test-00003-of-00013.parquet: downloading bytes:           |  0.00B            

{0: 300, 1: 300}
M1: AUC 0.919, threshold 0.259 (real called real 84.0%, AI detected 89.3%)
M2: AUC 0.970, threshold 0.346 (real called real 92.3%, AI detected 92.0%)
mean: AUC 0.968, threshold 0.386 (real called real 93.3%, AI detected 89.0%)


490

## Independent check on the WhatsApp images

Full-image scores of the 100 WhatsApp pairs with the calibrated thresholds (these images were not used for calibration).

In [5]:
pairs = pd.read_csv(ROOT / "results" / "e6_whatsapp_predictions.csv")
check = []
for when, folder, column in [("before", ROOT / "whatsapp" / "original", "file"), ("after", ROOT / "whatsapp" / "received", "received_file")]:
    scores = pd.DataFrame([full_image_scores(Image.open(folder / f)) for f in pairs[column]])
    for m in ["M1", "M2", "mean"]:
        decided = scores[m].to_numpy() >= thresholds[m]
        y = pairs["y"].to_numpy()
        check.append({"when": when, "detector": m, "auc": roc_auc_score(y, scores[m]),
                      "real_called_real": float((~decided[y == 0]).mean()), "ai_detected": float(decided[y == 1].mean())})
check = pd.DataFrame(check)
check.to_csv(ROOT / "results" / "website_whatsapp_check.csv", index=False)
display(check.round(3))

,when,detector,auc,real_called_real,ai_detected
0,before,M1,0.871,0.82,0.82
1,before,M2,0.953,0.88,0.92
2,before,mean,0.942,0.90,0.74
3,after,M1,0.751,0.76,0.58
4,after,M2,0.933,0.94,0.58
5,after,mean,0.898,0.98,0.48


## Upload

In [6]:
api = HfApi(token=getpass("Hugging Face write token: "))
api.create_repo(REPO, repo_type="model", private=PRIVATE, exist_ok=True)
api.upload_folder(folder_path=OUT, repo_id=REPO, repo_type="model", commit_message="Website models")
print(f"Uploaded to https://huggingface.co/{REPO}")

Hugging Face write token: ··········
Uploaded to https://huggingface.co/tanish-237/ai-image-forensics-models
